# 5 — TF-IDF + Logistic Regression: Model Training and Comparison

## Objective

Build a **multi-label condition classification system** that accepts symptom text and ranks possible conditions. We will train **three Logistic Regression configurations**, compare their validation metrics, and save their model artifacts for later evaluation and API integration.

Our workflow:

1. Load the prepared, frozen training and validation sets.
2. Convert symptom text into TF-IDF features.
3. Train one-vs-rest Logistic Regression for 376 condition labels.
4. Tune each model's binary decision threshold **using validation only**.
5. Compare precision, recall, F1, and Top-1/3/5 rankings.
6. Save all three fitted models and their validation JSON reports.

**Scope:** These experiments use NHS-derived symptom combinations. They are not independent patient cases, and the scores do not establish clinical diagnostic accuracy. The **test set is not read or used** anywhere in this notebook.

## 1. Imports and project paths

Run the notebook from any folder inside the `medical-diagnosis` repository. Inputs live in `app/dataset/output/training`. Outputs will be saved **without per-version directories**:

```text
app/
├── model/
│   └── tfidf_logistic_regression/
│       ├── tfidf_logistic_regression_v1.joblib
│       ├── tfidf_logistic_regression_v2.joblib
│       └── tfidf_logistic_regression_v3.joblib
└── dataset/output/training/
    ├── train.json
    ├── validation.json
    ├── test.json                      (never loaded)
    └── tfidf_logistic_regression/
        ├── tfidf_logistic_regression_v1_validation_metrics.json
        ├── tfidf_logistic_regression_v2_validation_metrics.json
        ├── tfidf_logistic_regression_v3_validation_metrics.json
        ├── tfidf_logistic_regression_comparison.json
        └── tfidf_logistic_regression_comparison.csv
```

The notebook refuses to overwrite existing files. If you want a completely new run, move/rename old outputs **before executing the saving cell**.

In [ ]:
from __future__ import annotations

import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

RANDOM_SEED = 42

# Fingerprints of the frozen dataset used by this project.
EXPECTED_TRAIN_SHA256 = "292fc80ddc16cc5afa9b49d9b93263775118e14af3b3de2795db7d06f336a6e1"
EXPECTED_VAL_SHA256 = "c61e0a2ab6524829ddf29979de836c11623f40c1d3a10084af6a34cc0b340f19"


def find_project_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        if (parent / "app" / "dataset").is_dir():
            return parent
    raise FileNotFoundError("Open the notebook from inside the medical-diagnosis repository")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "app" / "dataset" / "output" / "training"
TRAIN_FILE = DATA_DIR / "train.json"
VAL_FILE = DATA_DIR / "validation.json"
MODEL_DIR = ROOT / "app" / "model" / "tfidf_logistic_regression"
REPORT_DIR = DATA_DIR / "tfidf_logistic_regression"

for path in (TRAIN_FILE, VAL_FILE):
    if not path.is_file():
        raise FileNotFoundError(f"Missing required dataset: {path}")

DATASET_HASHES = {
    "train_json_sha256": sha256_file(TRAIN_FILE),
    "validation_json_sha256": sha256_file(VAL_FILE),
}
if DATASET_HASHES["train_json_sha256"] != EXPECTED_TRAIN_SHA256:
    raise ValueError("Train dataset SHA-256 differs from the expected frozen dataset")
if DATASET_HASHES["validation_json_sha256"] != EXPECTED_VAL_SHA256:
    raise ValueError("Validation dataset SHA-256 differs from the expected frozen dataset")

print("Repository:", ROOT)
print("Training data:", TRAIN_FILE)
print("Validation data:", VAL_FILE)
print("Model output folder:", MODEL_DIR)
print("JSON output folder:", REPORT_DIR)
print("Dataset fingerprints verified. Test set is untouched.")

## 2. Load the data and encode multi-label targets

Each example contains normalized `symptoms_classical` text and a list of one or more `labels` (conditions). We fit a `MultiLabelBinarizer` on the **training labels** and reuse that exact label order for validation and all models.

We also verify that text and example IDs do not overlap across the two splits. TF-IDF vocabulary will be learned from **training text only**.

In [ ]:
def load_examples(path: Path) -> list[dict]:
    rows = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"Expected a nonempty JSON list in {path}")
    seen = set()
    for i, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(f"Invalid row {i} in {path.name}")
        text, labels, example_id = (
            row.get("symptoms_classical"), row.get("labels"), row.get("example_id")
        )
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f"Empty symptoms_classical at row {i}")
        if (not isinstance(labels, list) or not labels or not all(
            isinstance(label, str) and label.strip() for label in labels
        )):
            raise ValueError(f"Invalid labels at row {i}")
        if not isinstance(example_id, str) or not example_id or example_id in seen:
            raise ValueError(f"Missing or repeated example_id at row {i}")
        seen.add(example_id)
    return rows


train_rows = load_examples(TRAIN_FILE)
val_rows = load_examples(VAL_FILE)
train_texts = [row["symptoms_classical"] for row in train_rows]
val_texts = [row["symptoms_classical"] for row in val_rows]
train_labels = [row["labels"] for row in train_rows]
val_labels = [row["labels"] for row in val_rows]

assert len(set(train_texts)) == len(train_rows), "Duplicate input text in training"
assert len(set(val_texts)) == len(val_rows), "Duplicate input text in validation"
assert not set(train_texts).intersection(val_texts), "Text overlap between splits"
assert not ({row["example_id"] for row in train_rows} &
            {row["example_id"] for row in val_rows}), "Example ID overlap"

label_binarizer = MultiLabelBinarizer()
y_train = label_binarizer.fit_transform(train_labels)  # preserve integer dtype
val_unknown = set(label for labels in val_labels for label in labels) - set(label_binarizer.classes_)
if val_unknown:
    raise ValueError(f"Validation has labels unseen during training: {sorted(val_unknown)}")
y_val = label_binarizer.transform(val_labels)
class_names = label_binarizer.classes_
supported = np.asarray(y_val.sum(axis=0) > 0).ravel()

# Match the original multi-label training behavior: do NOT convert y_train to uint8.
assert y_train.dtype.kind == "i" and y_val.dtype.kind == "i"
assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_val)).issubset({0, 1})
assert (y_train.sum(axis=1) >= 1).all() and (y_val.sum(axis=1) >= 1).all()
assert (len(train_rows), len(val_rows), len(class_names), int(supported.sum())) == (5494, 1177, 376, 324), (
    "Unexpected dataset dimensions: verify your frozen splits"
)

print(f"Training examples: {len(train_rows):,}")
print(f"Validation examples: {len(val_rows):,}")
print(f"Condition labels: {len(class_names):,}")
print(f"Classes appearing in validation: {int(supported.sum())}")
print(f"Multi-label training examples: {sum(len(x) > 1 for x in train_labels)}")
print(f"Target dtype: {y_train.dtype}")

## 3. Define the three candidate models

All candidates use TF-IDF features and an independent **one-vs-rest Logistic Regression** classifier. We vary the n-gram span, regularization strength `C`, and class balancing to study how they affect condition predictions.

| Model | Word n-grams | `C` | Class weighting | Hypothesis |
|---|---|---|---|---|
| **V1** | 1–3 | 20 | Balanced | Weaker regularization may capture distinctive symptom phrases |
| **V2** | 1–3 | 5 | None | Removing reweighting may improve disease ranking |
| **V3** | 1–4 | 10 | Balanced | Longer symptom phrases may improve contextual distinctions |

`C` is the inverse regularization strength: **higher C means weaker regularization**. Performance changes are empirical, not guaranteed. All models use the same training/validation examples, `max_features=50,000`, `sublinear_tf=True`, and `liblinear` solver.

In [ ]:
EXPERIMENTS = [
    {"version": "v1", "ngrams": (1, 3), "C": 20.0, "class_weight": "balanced",
     "description": "Word unigrams, bigrams and trigrams with C=20 and balanced weights"},
    {"version": "v2", "ngrams": (1, 3), "C": 5.0, "class_weight": None,
     "description": "Word unigrams, bigrams and trigrams with C=5 and no class weights"},
    {"version": "v3", "ngrams": (1, 4), "C": 10.0, "class_weight": "balanced",
     "description": "Word n-grams from length 1 to 4 with C=10 and balanced weights"},
]
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.025), 3).tolist()

display(pd.DataFrame([{
    "Model": row["version"].upper(), "Word n-grams": str(row["ngrams"]),
    "C": row["C"], "Class weight": str(row["class_weight"]),
} for row in EXPERIMENTS]))
print(f"Threshold candidates per model: {len(THRESHOLDS)}")

## 4. Evaluation functions

We will report two complementary kinds of validation performance:

- **Threshold-based multi-label metrics:** precision, recall, Micro F1, macro F1 and weighted F1. A condition is predicted as positive if its model score exceeds a selected threshold.
- **Ranking metrics:** Top-1, Top-3, and Top-5 hit rate plus label recall at K. A Top-K hit means **at least one** ground-truth condition is ranked among the first K.

The threshold is selected **only from the validation set**, maximizing Micro F1. Ranking metrics do **not** depend on that threshold. Per-label scores are not clinically calibrated probabilities.

In [ ]:
def ranking_metrics(true_binary: np.ndarray, scores: np.ndarray, k: int) -> dict:
    k = min(k, scores.shape[1])
    top_idx = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    top_true = np.take_along_axis(true_binary, top_idx, axis=1)
    label_counts = true_binary.sum(axis=1)
    assert (label_counts > 0).all()
    return {
        "top_k": int(k),
        "hit_rate": float((top_true.sum(axis=1) > 0).mean()),
        "label_recall_at_k": float(np.mean(top_true.sum(axis=1) / label_counts)),
    }


def evaluate_model(scores: np.ndarray, thresholds: list[float]):
    scores = np.asarray(scores, dtype=np.float64)
    assert scores.shape == y_val.shape, "Prediction array / labels shape mismatch"
    assert np.isfinite(scores).all(), "Non-finite prediction scores"
    assert ((scores >= 0) & (scores <= 1)).all(), "Scores must be between 0 and 1"

    threshold_rows = []
    for threshold in thresholds:
        predicted = (scores >= threshold).astype(np.uint8)
        threshold_rows.append({
            "threshold": float(threshold),
            "micro_f1": float(f1_score(y_val, predicted, average="micro", zero_division=0)),
            "micro_precision": float(precision_score(y_val, predicted, average="micro", zero_division=0)),
            "micro_recall": float(recall_score(y_val, predicted, average="micro", zero_division=0)),
        })

    chosen = max(threshold_rows, key=lambda item: (item["micro_f1"], -item["threshold"]))
    best_threshold = chosen["threshold"]
    predicted = (scores >= best_threshold).astype(np.uint8)
    metrics = {
        "selected_threshold": best_threshold,
        "micro_precision": float(precision_score(y_val, predicted, average="micro", zero_division=0)),
        "micro_recall": float(recall_score(y_val, predicted, average="micro", zero_division=0)),
        "micro_f1": float(f1_score(y_val, predicted, average="micro", zero_division=0)),
        "macro_f1_all_classes": float(f1_score(y_val, predicted, average="macro", zero_division=0)),
        "macro_f1_validation_supported_classes": float(f1_score(
            y_val[:, supported], predicted[:, supported], average="macro", zero_division=0
        )),
        "weighted_f1": float(f1_score(y_val, predicted, average="weighted", zero_division=0)),
        "top_k": {f"top_{k}": ranking_metrics(y_val, scores, k) for k in (1, 3, 5)},
    }
    return metrics, threshold_rows


def train_candidate(settings: dict):
    # Fit only on the training set; do not leak validation vocabulary.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=settings["ngrams"],
        min_df=1,
        max_features=50_000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    x_train = vectorizer.fit_transform(train_texts)
    x_val = vectorizer.transform(val_texts)

    estimator = LogisticRegression(
        solver="liblinear",
        C=settings["C"],
        max_iter=1000,
        class_weight=settings["class_weight"],
        random_state=RANDOM_SEED,
    )
    classifier = OneVsRestClassifier(estimator, n_jobs=1)
    started = time.perf_counter()
    classifier.fit(x_train, y_train)
    seconds = time.perf_counter() - started
    scores = np.asarray(classifier.predict_proba(x_val), dtype=np.float64)
    return vectorizer, classifier, scores, seconds

## 5. Train V1, V2 and V3

Each candidate has its **own freshly fitted vectorizer and classifier**. You can run all three in the same kernel: training them sequentially does not transfer weights or vocabulary between models.

We'll print Micro F1, Top-1, and Top-5 after each run. A sanity check stops the notebook if the model is behaving implausibly poorly on this particular prepared validation set; this catches label-order mistakes before anything is saved.

In [ ]:
results = []
for settings in EXPERIMENTS:
    version = settings["version"]
    print(f"\nTraining {version.upper()}: {settings['description']}", flush=True)
    vectorizer, classifier, scores, training_seconds = train_candidate(settings)
    metrics, threshold_rows = evaluate_model(scores, THRESHOLDS)

    top1 = metrics["top_k"]["top_1"]["hit_rate"]
    top5 = metrics["top_k"]["top_5"]["hit_rate"]
    print(f"  Training: {training_seconds:.1f} s | Vocab: {len(vectorizer.vocabulary_):,}")
    print(f"  Micro F1: {metrics['micro_f1']:.2%} | Top-1: {top1:.2%} | Top-5: {top5:.2%}")
    print(f"  Best threshold: {metrics['selected_threshold']:.3f}", flush=True)

    # Dataset-specific sanity guard against a broken label/score alignment.
    if metrics["micro_f1"] < 0.50 or top1 < 0.50:
        raise RuntimeError(
            f"Unexpectedly low {version} validation results. STOP: check training "
            "labels, dataset order and model output alignment before saving."
        )

    results.append({
        "version": version,
        "settings": settings,
        "vectorizer": vectorizer,
        "classifier": classifier,
        "metrics": metrics,
        "threshold_search": threshold_rows,
        "training_seconds": training_seconds,
        "scores": scores,
    })

assert [item["version"] for item in results] == ["v1", "v2", "v3"]
print("\nAll three models trained and validated successfully.")

## 6. Compare all three validation results

**Micro F1** is the primary selection metric for the multi-label classifier. We also display Top-K accuracy because the system will rank possible conditions. The best ranking model may differ from the highest-F1 model.

If two models are tied on Micro F1, we break ties with Top-5, then Top-1, then macro F1 on classes represented in validation. **No test-set information is used for selection.**

In [ ]:
comparison_rows = []
for item in results:
    m, e = item["metrics"], item["settings"]
    comparison_rows.append({
        "model": item["version"],
        "word_ngrams": str(e["ngrams"]),
        "C": float(e["C"]),
        "class_weight": str(e["class_weight"]),
        "threshold": m["selected_threshold"],
        "micro_precision": m["micro_precision"],
        "micro_recall": m["micro_recall"],
        "micro_f1": m["micro_f1"],
        "macro_f1_supported": m["macro_f1_validation_supported_classes"],
        "weighted_f1": m["weighted_f1"],
        "top_1": m["top_k"]["top_1"]["hit_rate"],
        "top_3": m["top_k"]["top_3"]["hit_rate"],
        "top_5": m["top_k"]["top_5"]["hit_rate"],
        "training_seconds": item["training_seconds"],
    })

comparison_df = pd.DataFrame(comparison_rows)
ranking_df = comparison_df.sort_values(
    ["micro_f1", "top_5", "top_1", "macro_f1_supported"],
    ascending=[False] * 4,
).reset_index(drop=True)
BEST_VERSION = ranking_df.iloc[0]["model"]
RANKING_VERSION = comparison_df.sort_values(
    ["top_5", "top_1", "micro_f1"], ascending=[False] * 3
).iloc[0]["model"]

print("Best multi-label Micro F1:", BEST_VERSION.upper())
print("Best Top-5 ranking:", RANKING_VERSION.upper())
display(ranking_df.style.format({
    "threshold": "{:.3f}",
    "micro_precision": "{:.2%}", "micro_recall": "{:.2%}",
    "micro_f1": "{:.2%}", "macro_f1_supported": "{:.2%}",
    "weighted_f1": "{:.2%}", "top_1": "{:.2%}",
    "top_3": "{:.2%}", "top_5": "{:.2%}",
    "training_seconds": "{:.1f}",
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
labels = [m.upper() for m in comparison_df["model"]]
axes[0].bar(labels, comparison_df["micro_f1"] * 100)
axes[0].set_ylim(0, 100)
axes[0].set_ylabel("Validation Micro F1 (%)")
axes[0].set_title("Micro F1 comparison")

x = np.arange(len(labels))
width = 0.25
for offset, key, label in [(-width, "top_1", "Top-1"), (0, "top_3", "Top-3"), (width, "top_5", "Top-5")]:
    axes[1].bar(x + offset, comparison_df[key] * 100, width=width, label=label)
axes[1].set_xticks(x, labels)
axes[1].set_ylim(0, 105)
axes[1].set_ylabel("Validation hit rate (%)")
axes[1].set_title("Condition ranking comparison")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Save three fitted models and flat validation reports

Each saved `.joblib` bundle includes the TF-IDF vectorizer, one-vs-rest classifier, consistent label encoder, selected threshold and metadata. Each version gets one JSON validation report.

**The reports go directly inside `tfidf_logistic_regression/`, without V1/V2/V3 folders.** Models likewise share one flat directory. The notebook checks for filename collisions and refuses to overwrite previous artifacts.

The reports record the **actual fitted hyperparameters**, not manually typed values. They also include dataset fingerprints to make comparisons reproducible. Only load `.joblib` files from sources you trust.

In [ ]:
if [item["version"] for item in results] != ["v1", "v2", "v3"]:
    raise RuntimeError("All three models must be trained before saving")

created_at = datetime.now(timezone.utc).isoformat()
model_paths = {item["version"]: MODEL_DIR / f"tfidf_logistic_regression_{item['version']}.joblib"
               for item in results}
report_paths = {item["version"]: REPORT_DIR / f"tfidf_logistic_regression_{item['version']}_validation_metrics.json"
                for item in results}
summary_json = REPORT_DIR / "tfidf_logistic_regression_comparison.json"
summary_csv = REPORT_DIR / "tfidf_logistic_regression_comparison.csv"

# Validate all destination paths before writing any outputs.
all_paths = list(model_paths.values()) + list(report_paths.values()) + [summary_json, summary_csv]
existing = [str(path) for path in all_paths if path.exists()]
if existing:
    raise FileExistsError(
        "Outputs already exist. Move or rename them before saving a fresh run; nothing was overwritten:\n"
        + "\n".join(existing)
    )

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for item in results:
    version = item["version"]
    vectorizer, classifier = item["vectorizer"], item["classifier"]
    model_name = f"tfidf_logistic_regression_{version}"
    metadata = {
        "name": model_name,
        "type": "TF-IDF + Logistic Regression (One-vs-Rest multi-label)",
        "createdAt": created_at,
        "training_examples": len(train_rows),
        "validation_examples": len(val_rows),
        "class_count": len(class_names),
        "validation_supported_class_count": int(supported.sum()),
        "validation_missing_class_count": int((~supported).sum()),
        "training_seconds": round(item["training_seconds"], 3),
        "scikit_learn_version": sklearn.__version__,
        "random_seed": RANDOM_SEED,
        "features": "symptoms_classical",
        "tfidf": {
            "kind": "word",
            "analyzer": vectorizer.analyzer,
            "ngram_range": list(vectorizer.ngram_range),
            "min_df": vectorizer.min_df,
            "max_features": vectorizer.max_features,
            "sublinear_tf": vectorizer.sublinear_tf,
            "vocabulary_size": len(vectorizer.vocabulary_),
        },
        "classifier": {
            "solver": classifier.estimator.solver,
            "C": float(classifier.estimator.C),
            "class_weight": classifier.estimator.class_weight,
            "max_iter": classifier.estimator.max_iter,
        },
        "dataset_sha256": DATASET_HASHES,
        "evaluation_scope": "NHS-derived symptom subsets; validation only; no test access",
        "probability_note": "Scores are not clinically calibrated disease probabilities",
    }
    bundle = {
        "vectorizer": vectorizer,
        "classifier": classifier,
        "label_binarizer": label_binarizer,
        "threshold": item["metrics"]["selected_threshold"],
        "metadata": metadata,
    }
    model_file = model_paths[version]
    metrics_file = report_paths[version]
    joblib.dump(bundle, model_file, compress=3)
    report = {
        "model": model_name,
        "metadata": metadata,
        "validation_metrics": item["metrics"],
        "threshold_search": item["threshold_search"],
        "test_set_used": False,
    }
    metrics_file.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    # Verify the saved model can reload and make valid predictions.
    loaded = joblib.load(model_file)  # trusted file created in this cell
    assert np.array_equal(loaded["label_binarizer"].classes_, class_names)
    assert loaded["vectorizer"].ngram_range == item["settings"]["ngrams"]
    assert loaded["classifier"].estimator.C == item["settings"]["C"]
    assert loaded["classifier"].estimator.class_weight == item["settings"]["class_weight"]
    assert loaded["threshold"] == item["metrics"]["selected_threshold"]
    check_scores = loaded["classifier"].predict_proba(
        loaded["vectorizer"].transform([val_texts[0]])
    )[0]
    assert len(check_scores) == len(class_names) and np.isfinite(check_scores).all()
    print("Saved model:", model_file.name)
    print("Saved metrics:", metrics_file.name)

comparison_df.to_csv(summary_csv, index=False)
comparison_summary = {
    "createdAt": created_at,
    "selected_model_for_micro_f1": f"tfidf_logistic_regression_{BEST_VERSION}",
    "selected_model_for_top_5": f"tfidf_logistic_regression_{RANKING_VERSION}",
    "selection_metric": "validation Micro F1",
    "dataset_sha256": DATASET_HASHES,
    "training_examples": len(train_rows),
    "validation_examples": len(val_rows),
    "class_count": len(class_names),
    "test_set_used": False,
    "comparison": comparison_df.to_dict(orient="records"),
}
summary_json.write_text(json.dumps(comparison_summary, indent=2), encoding="utf-8")

print("\nAll artifacts saved successfully.")
print("Models:", MODEL_DIR)
print("JSON reports and comparison:", REPORT_DIR)

## 8. Inspect predictions from the selected model

Use a short free-text symptom description to inspect the highest-scoring condition labels. Scores can rank predictions, but are **not** estimates of the actual probability that a patient has a particular disease.

In [ ]:
best = next(item for item in results if item["version"] == BEST_VERSION)


def rank_conditions(symptoms: str, top_k: int = 5) -> pd.DataFrame:
    if not isinstance(symptoms, str) or not symptoms.strip():
        raise ValueError("Enter a nonempty symptom description")
    if top_k < 1:
        raise ValueError("top_k must be at least 1")
    features = best["vectorizer"].transform([symptoms])
    scores = best["classifier"].predict_proba(features)[0]
    idx = np.argsort(-scores, kind="stable")[:min(top_k, len(class_names))]
    return pd.DataFrame({
        "condition": class_names[idx],
        "model_score": np.round(scores[idx], 4),
    })

rank_conditions("persistent coughing, wheezing, and difficulty breathing", top_k=5)

## 9. Review model limitations and next steps

**Validation results only.** The prepared examples are derived from condition descriptions published by NHS Inform. Different examples can share source language, so held-out scores on these generated combinations can be optimistic relative to original patient wording. Some rare conditions do not appear in the validation data.

Before claiming strong generalization, evaluate independent symptom descriptions and clinically important/emergency conditions. Once model selection is complete, use the untouched **test set** for final reporting. Future RNN, LSTM and BioBERT models should reuse the same splits for comparison.

This is an **educational research prototype**, not a diagnostic medical device. Do not interpret raw classifier scores as clinically validated probabilities.